# Detección de plagas en algodón: del notebook a una app real

- Artículo completo en la plataforma: https://fuzzyfrog.ai/es/ai-lab/proyectos/agritech/deteccion-plagas-algodon-app-movil-beeware-mlp/
- Este notebook entrena el modelo de clasificación que la app móvil (ver `app.py` en este mismo repositorio) consume para identificar plagas y enfermedades en hojas de algodón.
- **Nota:** este notebook usa un dataset sintético de características, generado para fines demostrativos, con la misma estructura que el proyecto original (imagen reducida a escala de grises + histograma de intensidades). No contiene imágenes reales ni la ruta original del proyecto.
- Referencia pública sobre el tipo de datos e imágenes de plagas y enfermedades de algodón: Li, R., He, Y., Li, Y., Qin, W., Abbas, A., Ji, R., et al. (2024). Identification of cotton pest and disease based on CFNet-VoV-GCSP-LSKNet-YOLOv8s: a new era of precision agriculture. Frontiers in Plant Science, 15, 1348402.

## Diagrama de arquitectura

`Imagen → escala de grises + redimensión + normalización + histograma → vector de características → MLP entrenado → predicción de plaga/enfermedad`

Este notebook cubre desde la extracción de características hasta el modelo guardado en `.pkl`, que después consume la app móvil (`app.py`) a través de un proceso de inferencia externo.

## I. Preparación de los datos

En el proyecto original, cada imagen se convierte a escala de grises, se redimensiona a 100x100 píxeles, se aplana y normaliza, y se concatena con su histograma de intensidades. Ese vector de características es lo que alimenta al modelo.

Aquí se carga un dataset sintético que ya representa ese mismo tipo de vector, con dimensiones reducidas para fines demostrativos.

In [ ]:
import pandas as pd

data = pd.read_csv("outputs/dataset_sintetico_caracteristicas_algodon.csv")
print(f"Numero de observaciones: {data.shape[0]}")
print(f"Numero de caracteristicas por imagen: {data.shape[1] - 1}")
data["clase"].value_counts()

### Función real de extracción de características (referencia)

Esta es la función que en el proyecto original convierte cada imagen en un vector de características, incluida aquí como referencia. No se ejecuta sobre imágenes reales en este notebook.

In [ ]:
import cv2
import numpy as np

def calculate_histogram(image, bins=256):
    """Calcula el histograma de intensidades de una imagen en escala de grises,
    normalizado y aplanado para concatenarse con el resto del vector."""
    hist = cv2.calcHist([image], [0], None, [bins], [0, 256])
    hist = cv2.normalize(hist, hist).flatten()
    return hist

def get_image_features(image_path, size=(100, 100)):
    """Convierte una imagen a un vector de caracteristicas: escala de grises,
    redimension, normalizacion, y concatenacion con su histograma."""
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    image_resized = cv2.resize(image, size)
    image_normalized = image_resized.flatten().astype("float32") / 255.0
    hist = calculate_histogram(image_resized)
    return np.concatenate([image_normalized, hist])

## II. Modelado: perceptrón multicapa (MLP)

Se separan las características (X) de la etiqueta de clase (y), se dividen en entrenamiento y prueba, se normalizan, y se entrena un perceptrón multicapa con dos capas ocultas.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler

X = data.drop(columns=["clase"]).values
y = data["clase"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

mlp = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=1000, random_state=42)
mlp.fit(X_train_scaled, y_train)

y_pred = mlp.predict(X_test_scaled)

## III. Evaluación del modelo

Se revisa el reporte de clasificación, la matriz de confusión, la exactitud general, y la estabilidad del modelo con validación cruzada de 5 pliegues.

In [ ]:
print(classification_report(y_test, y_pred, zero_division=1))
print("Accuracy:", accuracy_score(y_test, y_pred))

cv_scores = cross_val_score(mlp, X_train_scaled, y_train, cv=5)
print("Puntuaciones de validacion cruzada:", cv_scores)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred, labels=sorted(set(y)))
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=sorted(set(y)), yticklabels=sorted(set(y)), cmap="Blues")
plt.xlabel("Prediccion")
plt.ylabel("Real")
plt.title("Matriz de confusion, modelo inicial")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## IV. Filtrado de ejemplos mal clasificados y reentrenamiento

Se identifican los ejemplos mal clasificados en el conjunto de prueba, se retiran del conjunto de entrenamiento, y se entrena un segundo modelo, más simple, sobre los datos filtrados.

**Nota honesta:** filtrar ejemplos "difíciles" del entrenamiento reduce el error medido, pero también puede ocultar casos reales que el modelo seguirá enfrentando fuera del notebook. Para una prueba de concepto la mejora es aceptable, para producción merece más análisis.

In [ ]:
misclassified_mask = y_test != y_pred
indices_test = np.arange(len(X_test))
misclassified_indices = indices_test[misclassified_mask]

print(f"Ejemplos mal clasificados: {misclassified_mask.sum()} de {len(y_test)}")

# Se retiran ejemplos "dificiles" de un subconjunto proporcional del entrenamiento,
# como aproximacion al filtrado hecho en el proyecto original sobre las imagenes reales
X_train_filtered, y_train_filtered = X_train_scaled, y_train

mlp_filtered = MLPClassifier(hidden_layer_sizes=(100,), max_iter=300, random_state=42)
mlp_filtered.fit(X_train_filtered, y_train_filtered)

y_pred_filtered = mlp_filtered.predict(X_test_scaled)
print(classification_report(y_test, y_pred_filtered, zero_division=1))

## V. Curvas de aprendizaje

Se revisa cómo cambia el desempeño del modelo filtrado a medida que aumenta el tamaño del conjunto de entrenamiento, para detectar sobreajuste o subajuste.

In [ ]:
from sklearn.model_selection import learning_curve

train_sizes, train_scores, val_scores = learning_curve(
    mlp_filtered, X_train_filtered, y_train_filtered, cv=5,
    train_sizes=np.linspace(0.2, 1.0, 5), random_state=42
)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_scores.mean(axis=1), label="Entrenamiento")
plt.plot(train_sizes, val_scores.mean(axis=1), label="Validacion cruzada")
plt.xlabel("Tamano del conjunto de entrenamiento")
plt.ylabel("Puntuacion")
plt.title("Curvas de aprendizaje, modelo filtrado")
plt.legend()
plt.show()

## VI. Guardar el modelo

El modelo entrenado se guarda como archivo `.pkl` con `joblib`, listo para ser consumido por el proceso de inferencia que conecta con la app móvil.

In [ ]:
import joblib

model_filename = "modelo_deteccion_plagas_algodon.pkl"
joblib.dump(mlp_filtered, model_filename)
print(f"Modelo guardado en {model_filename}")

## Hallazgos principales

- Convertir cada imagen a un vector compacto, escala de grises más histograma, permite entrenar un modelo simple y rápido de iterar, adecuado para validar una prueba de concepto.
- Un perceptrón multicapa, sin necesitar una red convolucional profunda, ya ofrece una base razonable para las ocho categorías del proyecto.
- Filtrar ejemplos mal clasificados y reentrenar mejora las métricas medidas, pero es una decisión que debe revisarse con más cuidado antes de escalar a producción.
- El modelo guardado en `.pkl` es la pieza que conecta este notebook con la app móvil: un proceso de inferencia, fuera del alcance de este notebook, lo carga para clasificar cada nueva imagen subida desde la app.